# AC-DC Converters (Rectifiers)

This chapter demonstrates theoretical and simulated waveforms for major rectifier topologies using the `power_sim` package.

Topics covered:
- Single-phase half-wave and full-wave bridge
- Three-phase half-wave and six-pulse bridge
- Uncontrolled diode and controlled SCR operation with firing angle $\alpha$

## Key Theory

For a single-phase source $v_s = V_m\sin\theta$:

1. Half-wave diode rectifier ($\alpha=0$):
$$V_{dc}=\frac{V_m}{\pi}, \quad V_{rms}=\frac{V_m}{2}$$

2. Half-wave controlled rectifier (R-load): conduction from $\theta=\alpha$ to $\pi$:
$$V_{dc}=\frac{V_m}{2\pi}(1+\cos\alpha)$$
$$V_{rms}=V_m\sqrt{\frac{\pi-\alpha}{4\pi}+\frac{\sin(2\alpha)}{8\pi}}$$

3. Full-wave bridge diode rectifier:
$$V_{dc}=\frac{2V_m}{\pi}, \quad V_{rms}=\frac{V_m}{\sqrt{2}}$$

4. Fully controlled bridge with continuous current:
$$V_{dc}=\frac{2V_m}{\pi}\cos\alpha$$

For a three-phase six-pulse bridge using line-line peak $V_{LL,pk}$:
$$V_{dc}=\frac{3V_{LL,pk}}{\pi}\cos\alpha$$
(with $\alpha=0$ for diode bridge).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from power_sim import ac_dc, loads, plotter, utils

In [ ]:
# Timebase for one fundamental cycle
f = 50.0
time = utils.timebase(fundamental_frequency=f, cycles=1.0, samples_per_cycle=4000)
omega_t = plotter.to_omega_t(time, f)
Vm = 325.0  # ~230 Vrms source peak

## Single-Phase Half-Wave: Diode vs Controlled SCR

Conduction logic:
- Diode case: conducts only for positive source half-cycle.
- SCR case (R-load): after trigger at $\alpha$, conduction continues until natural current zero at $\pi$.

In [ ]:
alpha_deg = 60.0
alpha = np.deg2rad(alpha_deg)

half_diode = ac_dc.single_phase_half_wave_rectifier(time, source_peak=Vm, frequency=f, controlled=False)
half_scr = ac_dc.single_phase_half_wave_rectifier(time, source_peak=Vm, frequency=f, alpha=alpha, controlled=True)

print('Half-wave diode theory:', half_diode['theory'])
print('Half-wave diode simulated metrics:', half_diode['metrics'])
print(f'Half-wave SCR theory (alpha={alpha_deg:.1f} deg):', half_scr['theory'])
print('Half-wave SCR simulated metrics:', half_scr['metrics'])

In [ ]:
fig, _ = plotter.plot_stacked_waveforms(
    x=omega_t,
    traces=[
        {'y': half_diode['source_voltage'], 'label': 'Source voltage', 'ylabel': 'v_s (V)'},
        {'y': half_diode['output_voltage'], 'label': 'Half-wave diode output', 'ylabel': 'v_o (V)'},
        {'y': half_scr['output_voltage'], 'label': f'Half-wave SCR output, alpha={alpha_deg:.0f} deg', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Electrical angle, omega t (rad)',
    title='Single-Phase Half-Wave Rectifier Waveforms'
)
plt.show()

## Single-Phase Full-Bridge: Diode and Controlled

Bridge output is the absolute value of source in diode mode.
For controlled bridge with continuous current, output average follows $V_{dc}=\frac{2V_m}{\pi}\cos\alpha$.

In [ ]:
full_diode = ac_dc.single_phase_full_bridge_rectifier(time, source_peak=Vm, frequency=f, controlled=False)
full_scr_ccm = ac_dc.single_phase_full_bridge_rectifier(
    time, source_peak=Vm, frequency=f, alpha=alpha, controlled=True, continuous_current=True
)

print('Full-bridge diode theory:', full_diode['theory'])
print('Full-bridge diode simulated metrics:', full_diode['metrics'])
print(f'Full-bridge SCR CCM theory (alpha={alpha_deg:.1f} deg):', full_scr_ccm['theory'])
print('Full-bridge SCR CCM simulated metrics:', full_scr_ccm['metrics'])

In [ ]:
rl = loads.RLLoad(resistance=20.0, inductance=0.08, allow_negative_current=False)
rl_wave = rl.simulate(time, full_diode['output_voltage'], initial_current=0.0, settle_cycles=5)

fig, _ = plotter.plot_voltage_current(
    x=omega_t,
    voltage=rl_wave.voltage,
    current=rl_wave.current,
    xlabel='Electrical angle, omega t (rad)',
    voltage_label='Bridge output voltage (V)',
    current_label='RL load current (A)',
    title='Single-Phase Bridge Rectifier Feeding RL Load'
)
plt.show()
print({'average_current': rl_wave.average_current, 'rms_current': rl_wave.rms_current})

## Three-Phase Rectifiers

- Half-wave: the highest phase-to-neutral voltage conducts.
- Six-pulse bridge: one top and one bottom device conduct; output is selected line-line segment.

In [ ]:
three_half = ac_dc.three_phase_half_wave_rectifier(time, phase_peak=Vm, frequency=f, controlled=False)
three_bridge = ac_dc.three_phase_bridge_rectifier(time, phase_peak=Vm, frequency=f, controlled=False)

print('Three-phase half-wave theory:', three_half['theory'])
print('Three-phase bridge theory:', three_bridge['theory'])
print('Three-phase bridge simulated metrics:', three_bridge['metrics'])

In [ ]:
va, vb, vc = three_bridge['phase_voltages']
fig, _ = plotter.plot_stacked_waveforms(
    x=omega_t,
    traces=[
        {'y': va, 'label': 'v_a', 'ylabel': 'Phase V (V)'},
        {'y': vb, 'label': 'v_b', 'ylabel': 'Phase V (V)'},
        {'y': vc, 'label': 'v_c', 'ylabel': 'Phase V (V)'},
        {'y': three_bridge['output_voltage'], 'label': 'Bridge output', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Electrical angle, omega t (rad)',
    title='Three-Phase Six-Pulse Bridge Rectifier'
)
plt.show()